## Récupération des données Wikidata et enrichissement des fichiers XML et CSV d'index pour les personnes

On cherche ici à récupérer les informations relatives à chaque personne à partir de l'URL indiquée dans l'attribut @source de la balise <person>.
Pour chaque personne, on part de :  
```<person xml:id="PrenomNom" source="URL" role="activites"></person>```  pour arriver à :  

```
<person xml:id="PrenomNom" source="URL" role="activite">
  <persName>Prenom Nom
  <forename>Prenom</forename>
  <surname>Nom</surname>
  <addName>Variante du nom qui apparait dans le corpus</addName>
  <trait><p>desciption, éléments biographiques</p></trait>
  <birth date="date format ISO" place="lieu" />
  <death date="date format ISO" place="lieu" />
  <nationality>nationalité</nationality>
  <identifiers>
    <ulan>identifiant ULAN</ulan>
    <bnf>identifiant BnF</bnf>
    <viaf>identifiant VIAF</viaf>
    <isni>identifiant ISNI</isni>
    <vatican>identifiant Vatican</vatican>
    <wikidata>identifiant Wikidata</wikidata>
  </identifiers>
  <works>
    <work>Titre de l'oeuvre</work>
    <work>Titre de l'oeuvre</work>
  </works>
</person>
```
Cette structure est celle des nouvelles entrées, pour une partie conséquente des personnes, notamment les artistes la structure diffère un peu mais cela ne compromet pas le fonctionnement du script qui récupère et structure les données pour les nouveaux ajouts.  
Ensuite, les entrées ajoutées doivent être classées par ordre alphabétique parmi les entrées existantes.  
Enfin, on met à jour le fichier CSV. 

#### Import des librairies nécessaires

In [1]:
import xml.etree.ElementTree as ET
import requests
from time import sleep
from urllib.parse import urlparse
import csv
import os
import pandas as pd

#### Espace de nom TEI

In [2]:
ns = {"tei": "http://www.tei-c.org/ns/1.0"}
ET.register_namespace("", ns["tei"])

#### Chemin des fichiers XML et CSV

In [3]:
XML_FILE = "../../IndexArterm/IndexPersonnes.xml"
#XML_FILE = "test.xml"
CSV_FILE = "IndexPersonnes.csv"
VARIANTES_FILE = "../../Indexarterm/script-arterm/input-script/NER_balises/auteurs.csv"

In [ ]:
# HEADERS

HEADERS_WIKIDATA = {
            "User-Agent": "MyWikidataBot/1.0 (contact: pierrehusson482@gmail.com)"
        }

#### Définition des fonctions utiles

In [ ]:
#Récupération de l'identifiant QID à partir de l'URL Wikidata
def extract_qid(wikidata_url):
    return urlparse(wikidata_url).path.split("/")[-1]

# Fonction pour récupérer les informations d'une personne à partir de Wikidata
def get_wikidata_info(qid):
    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"
    r = requests.get(url, headers=HEADERS_WIKIDATA)
    if r.status_code != 200:
        return {}
    data = r.json()["entities"][qid]
    # Fonction pour extraire les valeurs de propriété dans une langue spécifique, ici le français
    def get_lang_value(prop, lang="fr"):
        try:
            return next(i["value"] for i in data["claims"][prop]
                        if "datavalue" in i["mainsnak"] and i["mainsnak"]["datavalue"]["value"].get("language", lang) == lang)
        except:
            return ""
        
    # Fonction pour récupérer le label d'une propriété dans une langue spécifique
    # (par exemple, le label de la personne en français) 
    def get_label(prop, lang="fr"):
        try:
            return data["labels"][lang]["value"]
        except:
            return ""
        
    # Fonction pour récupérer une valeur unique d'une propriété
    # (par exemple, date de naissance, lieu de naissance, etc.)
    def get_value(prop):
        try:
            return data["claims"][prop][0]["mainsnak"]["datavalue"]["value"]
        except:
            return None
        
    # Fonction pour récupérer plusieurs valeurs d'une propriété
    # (par exemple, occupations, œuvres, etc.)
    def get_multiple_labels(prop, lang="fr"):
        values = []
        for claim in data.get("claims", {}).get(prop, []):
            try:
                q = claim["mainsnak"]["datavalue"]["value"]["id"]
                label_url = f"https://www.wikidata.org/wiki/Special:EntityData/{q}.json"
                label_data = requests.get(label_url, headers=HEADERS_WIKIDATA).json()
                label = label_data["entities"][q]["labels"].get(lang, {}).get("value", "")
                if label:
                    values.append(label)
                sleep(0.1)
            except:
                continue
        return values

    label = get_label("fr") # Récupération du label en français
    description = get_lang_value("P31", "fr") or data.get("descriptions", {}).get("fr", {}).get("value", "") # Récupération de la description en français
    occupations = get_multiple_labels("P106", "fr") # Récupération des occupations en français
    nationality = get_multiple_labels("P27", "fr") # Récupération de la nationalité en français
    works = get_multiple_labels("P800", "fr") # Récupération de la liste des œuvres en français

    birth = get_value("P569") # Récupération de la date de naissance
    birth_date = birth["time"][1:11] if birth else "" # Récupération de la date de naissance au format AAAA-MM-JJ (norme ISO 8601)
    birth_place = get_multiple_labels("P19", "fr") # Récupération du lieu de naissance en français

    death = get_value("P570") # Récupération de la date de décès
    death_date = death["time"][1:11] if death else ""  # Récupération de la date de décès au format AAAA-MM-JJ (norme ISO 8601)
    death_place = get_multiple_labels("P20", "fr") # Récupération du lieu de décès en français

    # Fonction pour récupérer l'identifiant d'une propriété
    # (par exemple, P245 pour ULAN, P268 pour BNF, etc
    def get_id(prop):
        try:
            return data["claims"][prop][0]["mainsnak"]["datavalue"]["value"]
        except:
            return ""

    return {
        "label": label, # Récupération du label de la personne
        "forename": label.split()[0] if label else "", # Récupération du prénom de la personne
        "surname": label.split()[-1] if label else "", # Récupération du nom de famille de la personne
        "description": description, # Récupération de la description de la personne
        "occupations": occupations, # Récupération des occupations de la personne
        "nationality": nationality[0] if nationality else "", # Récupération de la nationalité de la personne
        "birth_date": birth_date, # Récupération de la date de naissance de la personne
        "birth_place": birth_place[0] if birth_place else "", # Récupération du lieu de naissance de la personne
        "death_date": death_date, # Récupération de la date de décès de la personne
        "death_place": death_place[0] if death_place else "", # Récupération du lieu de décès de la personne
        "ulan": get_id("P245"), # Récupération de l'identifiant ULAN
        "bnf": get_id("P268"), # Récupération de l'identifiant BNF
        "viaf": get_id("P214"), # Récupération de l'identifiant VIAF
        "isni": get_id("P213"), # Récupération de l'identifiant ISNI
        "vat": get_id("P8034"), # Récupération de l'identifiant VAT
        "wikidata": qid, # Récupération de l'identifiant Wikidata
        "works": works # Récupération de la liste des œuvres de la personne
    }

def needs_enrichment(person_el):
    if person_el.find("tei:nationality", ns) is None:
        return True
    ids = person_el.find("tei:identifiers", ns)
    if ids is None:
        return True
    for key in ["ulan", "bnf", "viaf", "isni", "vat", "wikidata"]:
        if ids.find(f"tei:{key}", ns) is None:
            return True
    works = person_el.find("tei:works", ns)
    if works is None or not works.findall("tei:work", ns):
        return True
    return False

def is_roman_numeral(text):
    import re
    return bool(re.fullmatch(r"[IVXLCDM]+", text.strip(), re.IGNORECASE))

# Charger les variantes depuis le CSV ===
variantes_df = pd.read_csv(
    VARIANTES_FILE,
    dtype=str,
    quoting=csv.QUOTE_ALL,
    engine="python"
).fillna("")

#Création d'un dictionnaire pour les variantes
variantes_dict = {
    row["ID"]: [v.strip() for v in row["Noms"].split(",") if v.strip()]
    for _, row in variantes_df.iterrows()
}

#### Charger et parser le fichier XML

In [5]:
xml_path = XML_FILE 
tree = ET.parse(XML_FILE)
root = tree.getroot()
listPerson = root.find(".//tei:listPerson", ns)

#### Enrichissement du fichier XML

In [6]:
all_persons = list(listPerson.findall("tei:person", ns))
for person in all_persons:
    source = person.get("source")
    if not source or not needs_enrichment(person):
        continue

    qid = extract_qid(source)
    info = get_wikidata_info(qid)
    sleep(1)

    if info["description"]:
        person.set("role", info["description"])
    
    xml_id = person.get("{http://www.w3.org/XML/1998/namespace}id")
    if xml_id and xml_id in variantes_dict:
        variantes = variantes_dict.get(xml_id)

    # Assurer la présence de <persName>
    persName = person.find("tei:persName", ns)
    if persName is None:
        persName = ET.SubElement(person, "persName")

    # Compléter <forename> si absent ou vide
    forename_el = persName.find("tei:forename", ns)
    if forename_el is None and info["forename"]:
        ET.SubElement(persName, "forename").text = info["forename"]
    '''elif forename_el is not None and (not forename_el.text or not forename_el.text.strip()) and info["forename"]:
        forename_el.text = info["forename"]'''

    # Compléter <surname> si absent ou vide, sauf si c'est un chiffre romain
    surname_el = persName.find("tei:surname", ns)
    if info["surname"] and not is_roman_numeral(info["surname"]):
        if surname_el is None:
            ET.SubElement(persName, "surname").text = info["surname"]
        elif not surname_el.text or not surname_el.text.strip():
            surname_el.text = info["surname"]

    # Récupérer les <addName> existants
    existing_variants = set(
        el.text.strip() for el in persName.findall("tei:addName", ns) if el.text
    )

    for variante in variantes:
        if variante not in existing_variants:
            add = ET.SubElement(persName, "addName")
            add.text = variante
            
    # birth
    birth = person.find("tei:birth", ns)
    if birth is None:
        birth = ET.SubElement(person, "birth")

    # Ajouter la date si non présente
    if not birth.get("when") and info["birth_date"]:
        birth.set("when", info["birth_date"])

    # Ajouter le lieu si non présent
    if birth.find("tei:placeName", ns) is None and info["birth_place"]:
        ET.SubElement(birth, "placeName").text = info["birth_place"]

    # death
    death = person.find("tei:death", ns)
    if death is None:
        death = ET.SubElement(person, "death")

    # Ajouter la date si non présente
    if not death.get("when") and info["death_date"]:
        death.set("when", info["death_date"])

    # Ajouter le lieu si non présent
    if death.find("tei:placeName", ns) is None and info["death_place"]:
        ET.SubElement(death, "placeName").text = info["death_place"]

    #nationality
    nat = person.find("tei:nationality", ns)
    if nat is None and info["nationality"]:
        ET.SubElement(person, "nationality").text = info["nationality"]

    # Liste des identifiants à ajouter (s'ils existent dans info)
    id_tags = ["ulan", "bnf", "viaf", "isni", "vat", "wikidata"]

    # Récupérer les identifiants déjà présents dans <person>
    existing = {
        (idno.get("type"), idno.text)
        for idno in person.findall("tei:idno", ns)
        if idno.get("type") and idno.text
    }

    # Ajouter les identifiants manquants
    for idtype in id_tags:
        value = info.get(idtype)
        if value and (idtype, value) not in existing:
            ET.SubElement(person, f"{{{ns['tei']}}}idno", {"type": idtype}).text = value

    #works
    works_el = person.find("tei:floruit", ns)
    if works_el is None and info["works"]:
        works_el = ET.SubElement(person, "floruit")
    if works_el is not None:
        existing_titles = set(w.text.strip() for w in works_el.findall("tei:objectName", ns) if w.text)
        for work in info["works"]:
            if work not in existing_titles:
                ET.SubElement(works_el, "objectName").text = work

#### Tri par ordre alphabétique et sauvegarde du fichier XML trié

In [7]:
# Tri des <person> par <surname>
def get_surname(person):
    sn_el = person.find(".//tei:surname", ns)
    if sn_el is not None and sn_el.text:
        return sn_el.text.strip().lower()
    return ""

sorted_persons = sorted(listPerson.findall("tei:person", ns), key=get_surname)

for person in listPerson.findall("tei:person", ns):
    listPerson.remove(person)
for person in sorted_persons:
    listPerson.append(person)

# Sauvegarde du XML trié
tree.write(XML_FILE, encoding="utf-8", xml_declaration=True)

#### Création du fichier CSV à partir du fichier XML

In [8]:
rows = []

for person in root.findall(".//tei:person", ns):
    person_data = {
        "xml:id": person.get("{http://www.w3.org/XML/1998/namespace}id"),
        "forename": "",
        "surname": "",
        "addName": "",
        "role": person.get("role", ""),
        "trait_texts": "",
        "birth_date": "",
        "birth_place": "",
        "death_date": "",
        "death_place": "",
        "nationality": "",
        "works": "",
        "ulan": "",
        "bnf": "",
        "viaf": "",
        "isni": "",
        "vat": "",
        "wikidata": ""
    }

    # persName complet
    persName = person.find("tei:persName", ns)
    if persName is not None:
        forename = persName.find("tei:forename", ns)
        surname = persName.find("tei:surname", ns)

        if forename is not None and forename.text:
            person_data["forename"] = forename.text.strip()

        if surname is not None and surname.text:
            person_data["surname"] = surname.text.strip()

        addNames = persName.findall("tei:addName", ns)
        person_data["addName"] = "; ".join([el.text.strip() for el in addNames if el.text])

    # Tous les <p> sous <trait>
    trait = person.find("tei:trait", ns)
    if trait is not None:
        p_texts = [p.text.strip() for p in trait.findall("tei:p", ns) if p.text]
        person_data["trait_texts"] = " | ".join(p_texts)

    # birth
    birth = person.find("tei:birth", ns)
    if birth is not None:
        person_data["birth_date"] = birth.get("date") or birth.get("when", "")
        person_data["birth_place"] = birth.get("place", "")
        if not person_data["birth_place"]:
            place_el = birth.find("tei:placeName", ns)
            if place_el is not None and place_el.text:
                person_data["birth_place"] = place_el.text.strip()

    # death
    death = person.find("tei:death", ns)
    if death is not None:
        person_data["death_date"] = death.get("date") or death.get("when", "")
        person_data["death_place"] = death.get("place", "")
        if not person_data["death_place"]:
            place_el = death.find("tei:placeName", ns)
            if place_el is not None and place_el.text:
                person_data["death_place"] = place_el.text.strip()


    # nationality
    nationality = person.find("tei:nationality", ns)
    if nationality is not None and nationality.text:
        person_data["nationality"] = nationality.text.strip()

    # works
    works = person.find("tei:works", ns)
    if works is not None:
        person_data["works"] = "; ".join(
            [w.text.strip() for w in works.findall("tei:work", ns) if w.text]
        )

    # identifiers
    identifiers = person.find("tei:identifiers", ns)
    if identifiers is not None:
        for ident in identifiers:
            tag = ident.tag.split("}")[-1]
            if tag in person_data and ident.text:
                person_data[tag] = ident.text.strip()

    rows.append(person_data)

#### Ecriture du fichier CSV, sans doublon

In [9]:
fieldnames = [
    "xml:id", "forename", "surname", "addName", "role", "trait_texts",
    "birth_date", "birth_place", "death_date", "death_place",
    "nationality", "works", "ulan", "bnf", "viaf", "isni", "vat", "wikidata"
]

with open(CSV_FILE, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for row in rows:
        writer.writerow(row)

print(f"✅ XML enrichi trié et enregistré dans '{XML_FILE}'")
print(f"✅ CSV généré trié dans '{CSV_FILE}'")

✅ XML enrichi trié et enregistré dans '../../IndexArterm/IndexPersonnes.xml'
✅ CSV généré trié dans 'IndexPersonnes.csv'
